# Scan_Orchestrator

Purpose: Batch runner for multiple named scan groups.

Executes multiple pre-configured scan batches in sequence, capturing status and timing,
and producing a summary report.

## Configuration and Setup

In [ ]:
import sys
import subprocess
import logging
from datetime import datetime
from typing import Dict, Tuple

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("[Orchestrator]")

## Editable Scan Batch Configuration

In [ ]:
# Editable scan batch configuration
SCAN_BATCHES = {
    "data_pipeline": "/Workspace/Users/sai_kalluri@bcbsil.com/CLOUD_BIS_AI_POC/notebooks/data",
    "ml_pipeline": "/Workspace/Users/sai_kalluri@bcbsil.com/CLOUD_BIS_AI_POC/notebooks/ml",
    "etl_jobs": "/Workspace/Shared/ETL",
    "reports": "/Workspace/Shared/Reports",
}

## Install Dependencies

In [ ]:
print("[Orchestrator] Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "/dbfs/user/sai_kalluri@bcbsil.com/CLOUD_BIS_AI_POC/BrickScanner/requirements.txt"])
dbutils.library.restartPython()

## Batch Execution Functions

In [ ]:
def run_batch_scan(batch_name: str, paths: str) -> Tuple[str, str, str]:
    """
    Run a single scan batch.
    
    Args:
        batch_name: Friendly name for batch
        paths: Comma-separated workspace paths
    
    Returns:
        Tuple of (batch_name, status, message)
        - status: "SUCCESS", "FAILED", "SKIPPED"
        - message: Status message
    """
    logger.info(f"[Orchestrator] Starting batch: {batch_name}")
    
    if not paths or not paths.strip():
        logger.warning(f"[Orchestrator] Batch {batch_name} has no paths, skipping")
        return (batch_name, "SKIPPED", "No paths configured")
    
    start_time = datetime.now()
    
    try:
        dbutils.notebook.run(
            "./Run_BrickScanner.ipynb",
            timeout_seconds=3600,
            arguments={
                "include_paths": paths,
                "dry_run": "false",
                "run_setup": "false"
            }
        )
        
        elapsed = (datetime.now() - start_time).total_seconds()
        message = f"Completed in {elapsed:.1f}s"
        logger.info(f"[Orchestrator] ✓ Batch {batch_name} succeeded: {message}")
        return (batch_name, "SUCCESS", message)
    
    except Exception as e:
        elapsed = (datetime.now() - start_time).total_seconds()
        message = f"Failed after {elapsed:.1f}s: {str(e)[:100]}"
        logger.error(f"[Orchestrator] ✗ Batch {batch_name} failed: {message}")
        return (batch_name, "FAILED", message)

## Summary Report Function

In [ ]:
def create_summary_table(results: list) -> None:
    """
    Display summary table of batch results.
    
    Args:
        results: List of (batch_name, status, message) tuples
    """
    import pandas as pd
    
    df = pd.DataFrame(results, columns=["Batch Name", "Status", "Message"])
    
    print("\n" + "="*80)
    print("SCAN_ORCHESTRATOR SUMMARY")
    print("="*80)
    display(spark.createDataFrame(df))
    
    # Summary stats
    total = len(results)
    succeeded = sum(1 for _, status, _ in results if status == "SUCCESS")
    failed = sum(1 for _, status, _ in results if status == "FAILED")
    skipped = sum(1 for _, status, _ in results if status == "SKIPPED")
    
    print(f"\nTotal batches: {total}")
    print(f"  ✓ Succeeded: {succeeded}")
    print(f"  ✗ Failed: {failed}")
    print(f"  ⊘ Skipped: {skipped}")
    print("="*80 + "\n")

## Main Execution

In [ ]:
# Main execution
try:
    print("[Orchestrator] Starting scan orchestration")
    print(f"[Orchestrator] Batches configured: {list(SCAN_BATCHES.keys())}")
    
    results = []
    
    for batch_name, paths in SCAN_BATCHES.items():
        batch_result = run_batch_scan(batch_name, paths)
        results.append(batch_result)
    
    # Display summary
    create_summary_table(results)
    
    # Check for failures
    failed_batches = [r for r in results if r[1] == "FAILED"]
    if failed_batches:
        print(f"[Orchestrator] ⚠ {len(failed_batches)} batch(es) failed")
    
    print("[Orchestrator] ✓ Orchestration completed")

except Exception as e:
    logger.error(f"Orchestration failed: {e}")
    raise